In [1]:
import torch
from PIL import Image
from transformers import Blip2Processor, Blip2ForConditionalGeneration, GPT2LMHeadModel, GPT2Tokenizer
import torchvision.transforms as T
import json
import os
from tqdm import tqdm
import torch.nn.functional as F

# Set device
device = "cuda" if torch.cuda.is_available() else "cpu"

# Load models and processors once
blip2_model_name = "Salesforce/blip2-flan-t5-xl"
processor = Blip2Processor.from_pretrained(blip2_model_name)
blip2_model = Blip2ForConditionalGeneration.from_pretrained(blip2_model_name, torch_dtype=torch.float16).to(device)

gpt2_model = GPT2LMHeadModel.from_pretrained("gpt2").to(device)
gpt2_tokenizer = GPT2Tokenizer.from_pretrained("gpt2")


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/432 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/21.0k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [2]:
# Generate refined caption function
def generate_refined_caption(image):
    if isinstance(image, str):
        img = Image.open(image).convert("RGB")
    else:
        img = image.convert("RGB")

    inputs = processor(images=img, return_tensors="pt").to(device)

    blip2_outputs = blip2_model.generate(**inputs)
    initial_caption = processor.decode(blip2_outputs[0], skip_special_tokens=True)

    input_text = f" {initial_caption}. Generate a more detailed or creative description."
    inputs_gpt = gpt2_tokenizer(input_text, return_tensors="pt").to(device)
    gpt_outputs = gpt2_model.generate(**inputs_gpt, max_length=100, num_return_sequences=1, pad_token_id=gpt2_tokenizer.eos_token_id)

    refined_text = gpt2_tokenizer.decode(gpt_outputs[0], skip_special_tokens=True)

    if '.' in refined_text:
        final_caption = refined_text.split('.')[0] + '.'
    else:
        final_caption = refined_text

    return final_caption


# FGSM Attack Function
def fgsm_attack(image_tensor, epsilon, data_grad):
    sign_data_grad = data_grad.sign()
    perturbed_image = image_tensor + epsilon * sign_data_grad
    perturbed_image = torch.clamp(perturbed_image, 0, 1)
    return perturbed_image


# PGD Attack Function
def pgd_attack(model, image_tensor, epsilon, alpha, num_iter):
    ori_image = image_tensor.clone().detach()
    perturbed_image = image_tensor.clone().detach()
    perturbed_image.requires_grad = True

    for _ in range(num_iter):
        # Forward pass through vision model
        vision_outputs = model.vision_model(pixel_values=perturbed_image.half())
        image_embeds = vision_outputs[0]

        # Dummy loss
        loss = image_embeds.norm()

        model.zero_grad()
        loss.backward()

        # Update image
        grad = perturbed_image.grad.data
        perturbed_image = perturbed_image + alpha * grad.sign()

        # Project back into epsilon-ball
        perturbation = torch.clamp(perturbed_image - ori_image, min=-epsilon, max=epsilon)
        perturbed_image = torch.clamp(ori_image + perturbation, 0, 1).detach_()
        perturbed_image.requires_grad = True

    return perturbed_image


# Carlini & Wagner Attack Function (simplified L2 version)
def cw_attack(model, image_tensor, targeted=False, c=1e-2, num_iter=100, learning_rate=1e-2):
    # Initialize perturbation
    perturbed_image = image_tensor.clone().detach()
    perturbed_image.requires_grad = True

    optimizer = torch.optim.Adam([perturbed_image], lr=learning_rate)

    ori_image = image_tensor.clone().detach()

    for _ in range(num_iter):
        optimizer.zero_grad()

        vision_outputs = model.vision_model(pixel_values=perturbed_image.half())
        perturbed_embeds = vision_outputs[0]

        vision_outputs_ori = model.vision_model(pixel_values=ori_image.half())
        ori_embeds = vision_outputs_ori[0]

        # L2 distance loss between original and perturbed embeddings
        l2_loss = F.mse_loss(perturbed_embeds, ori_embeds)

        # Total loss
        if targeted:
            loss = l2_loss  # For targeted attack
        else:
            loss = -l2_loss  # For untargeted attack

        total_loss = c * loss + F.mse_loss(perturbed_image, ori_image)
        total_loss.backward()
        optimizer.step()

        # Clamp to keep pixel values valid
        perturbed_image.data = torch.clamp(perturbed_image.data, 0, 1)

    return perturbed_image.detach()



# DeepFool Attack Function (simplified for image embeddings)
def deepfool_attack(model, image_tensor, num_iter=50, overshoot=0.02):
    perturbed_image = image_tensor.clone().detach()
    perturbed_image.requires_grad = True
    ori_image = image_tensor.clone().detach()

    for _ in range(num_iter):
        vision_outputs = model.vision_model(pixel_values=perturbed_image.half())
        perturbed_embeds = vision_outputs[0]

        loss = perturbed_embeds.norm()

        model.zero_grad()
        if perturbed_image.grad is not None:
            perturbed_image.grad.zero_()
        loss.backward()

        grad = perturbed_image.grad.data
        perturbation = overshoot * grad / grad.norm()
        perturbed_image = perturbed_image - perturbation
        perturbed_image = torch.clamp(perturbed_image, 0, 1).detach_()
        perturbed_image.requires_grad = True

    return perturbed_image



# Basic Iterative Method (BIM) Attack Function
def bim_attack(model, image_tensor, epsilon, alpha, num_iter):
    perturbed_image = image_tensor.clone().detach()
    perturbed_image.requires_grad = True
    ori_image = image_tensor.clone().detach()

    for _ in range(num_iter):
        vision_outputs = model.vision_model(pixel_values=perturbed_image.half())
        image_embeds = vision_outputs[0]

        loss = image_embeds.norm()

        model.zero_grad()
        loss.backward()

        grad = perturbed_image.grad.data
        perturbed_image = perturbed_image + alpha * grad.sign()

        # Project back into epsilon-ball
        perturbation = torch.clamp(perturbed_image - ori_image, min=-epsilon, max=epsilon)
        perturbed_image = torch.clamp(ori_image + perturbation, 0, 1).detach_()
        perturbed_image.requires_grad = True

    return perturbed_image



In [3]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [4]:

# Load the existing JSON annotations
input_json_path = "/content/drive/MyDrive/miniproject/subset_1k_data.json"  # update if needed
with open(input_json_path, 'r') as f:
    annotations = json.load(f)

# Folder where images are located
images_folder = "/content/drive/MyDrive/miniproject/Flicker8k_1kSubset/"

# Transformation for images
transform = T.Compose([
    T.Resize((processor.image_processor.size['height'], processor.image_processor.size['width'])),
    T.ToTensor()
])

# Attack parameters
epsilon = 0.09
alpha = 0.2
num_iter = 10

# Batch size
batch_size = 150

# Starting index (change this if you want to resume later)
start_idx = 850

# Output folder for batch results
output_folder = "/content/drive/MyDrive/miniproject/batch_results/"
os.makedirs(output_folder, exist_ok=True)

# Main loop
for batch_start in range(start_idx, len(annotations), batch_size):
    batch_end = min(batch_start + batch_size, len(annotations))
    batch = annotations[batch_start:batch_end]

    final_dataset = []

    print(f"Processing batch {batch_start} to {batch_end-1}")

    for item in tqdm(batch):
        image_name = item['image']
        ground_truth_captions = item['captions']
        image_path = os.path.join(images_folder, image_name)

        try:
            img = Image.open(image_path).convert("RGB")
        except Exception as e:
            print(f"Error loading {image_name}: {e}")
            continue

        # Generate original caption
        original_caption = generate_refined_caption(img)

        # Preprocess
        img_tensor = transform(img).unsqueeze(0).to(device)
        img_tensor.requires_grad = True

        # FGSM Attack
        vision_outputs = blip2_model.vision_model(pixel_values=img_tensor.half())
        image_embeds = vision_outputs[0]
        loss = image_embeds.norm()
        blip2_model.zero_grad()
        loss.backward()
        data_grad = img_tensor.grad.data
        fgsm_perturbed = fgsm_attack(img_tensor, epsilon, data_grad)
        fgsm_img = T.ToPILImage()(fgsm_perturbed.squeeze(0).cpu())
        fgsm_caption = generate_refined_caption(fgsm_img)

        # PGD Attack
        pgd_perturbed = pgd_attack(blip2_model, img_tensor, epsilon=epsilon, alpha=alpha, num_iter=num_iter)
        pgd_img = T.ToPILImage()(pgd_perturbed.squeeze(0).cpu())
        pgd_caption = generate_refined_caption(pgd_img)

        # CW Attack
        cw_perturbed = cw_attack(blip2_model, img_tensor, targeted=False, c=1e-1, num_iter=50, learning_rate=1e-2)
        cw_img = T.ToPILImage()(cw_perturbed.squeeze(0).cpu())
        cw_caption = generate_refined_caption(cw_img)

        # DeepFool Attack
        deepfool_perturbed = deepfool_attack(blip2_model, img_tensor, num_iter=30, overshoot=0.5)
        deepfool_img = T.ToPILImage()(deepfool_perturbed.squeeze(0).cpu())
        deepfool_caption = generate_refined_caption(deepfool_img)

        # BIM Attack
        bim_perturbed = bim_attack(blip2_model, img_tensor, epsilon=epsilon, alpha=alpha, num_iter=num_iter)
        bim_img = T.ToPILImage()(bim_perturbed.squeeze(0).cpu())
        bim_caption = generate_refined_caption(bim_img)

        # Store in dictionary
        result = {
            "image_path": image_path,
            "ground_truth_captions": ground_truth_captions,
            "original_caption": original_caption,
            "fgsm_caption": fgsm_caption,
            "pgd_caption": pgd_caption,
            "cw_caption": cw_caption,
            "deepfool_caption": deepfool_caption,
            "bim_caption": bim_caption
        }

        final_dataset.append(result)

    # Save the batch
    batch_output_path = os.path.join(output_folder, f"batch_{batch_start}_{batch_end-1}.json")
    with open(batch_output_path, 'w') as f:
        json.dump(final_dataset, f, indent=4)

    print(f"Saved batch {batch_start} to {batch_end-1} at {batch_output_path}")

print("✅ All batches processed!")


Processing batch 850 to 999


 13%|█▎        | 19/150 [09:55<1:08:14, 31.26s/it]/usr/local/lib/python3.11/dist-packages/torchvision/transforms/functional.py:282: RuntimeWarning: invalid value encountered in cast
  npimg = (npimg * 255).astype(np.uint8)
100%|██████████| 150/150 [1:17:20<00:00, 30.93s/it]

Saved batch 850 to 999 at /content/drive/MyDrive/miniproject/batch_results/batch_850_999.json
✅ All batches processed!


In [2]:
!pip install bert-score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 126.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 102.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 60.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 96.6 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalli

In [3]:
!git clone https://github.com/salaniz/pycocoevalcap

Cloning into 'pycocoevalcap'...
remote: Enumerating objects: 821, done.
remote: Counting objects: 100% (12/12), done.
remote: Compressing objects: 100% (9/9), done.
remote: Total 821 (delta 4), reused 3 (delta 3), pack-reused 809 (from 2)
Receiving objects: 100% (821/821), 130.06 MiB | 37.14 MiB/s, done.
Resolving deltas: 100% (424/424), done.


In [4]:

!pip install pycocoevalcap/


Processing ./pycocoevalcap
  Preparing metadata (setup.py) ... done
  Created wheel for pycocoevalcap: filename=pycocoevalcap-1.2-py3-none-any.whl size=104312245 sha256=5efa73af8f50449c97625059431d971bebcb3bd82320a2c3320738b21e8a02d1
  Stored in directory: /tmp/pip-ephem-wheel-cache-m69x0svu/wheels/0e/98/9f/b6578f2310a0adf702387edf950a2ba69dbf680c0b6830b312
Successfully built pycocoevalcap


In [5]:

import json
import numpy as np
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from pycocoevalcap.meteor.meteor import Meteor
from pycocoevalcap.cider.cider import Cider
from bert_score import score as bert_score


In [6]:

# Load your JSON file
with open("merged_file.json") as f:
    data = json.load(f)

smoother = SmoothingFunction().method4
meteor_scorer = Meteor()
cider_scorer = Cider()


In [11]:
# Initialize storage for scores
scores = {
    "original": {"BLEU": [], "METEOR": [], "CIDEr": []},
    "fgsm": {"BLEU": [], "METEOR": [], "CIDEr": [], "BERT": []},
    "pgd": {"BLEU": [], "METEOR": [], "CIDEr": [], "BERT": []},
    "cw": {"BLEU": [], "METEOR": [], "CIDEr": [], "BERT": []},
    "deepfool": {"BLEU": [], "METEOR": [], "CIDEr": [], "BERT": []},
    "bim": {"BLEU": [], "METEOR": [], "CIDEr": [], "BERT": []}
}

# List of attacks to evaluate
attacks = ["bim"]


In [12]:
def avg(lst): return round(np.mean(lst), 4)

# ORIGINAL CAPTION EVALUATION
print("\n🔍 Evaluating Original Captions...")
for idx, item in enumerate(data, 1):
    refs = [caption.lower().split() for caption in item["ground_truth_captions"]]
    orig = item["original_caption"].lower().split()

    scores["original"]["BLEU"].append(sentence_bleu(refs, orig, smoothing_function=smoother))
    scores["original"]["METEOR"].append(
        meteor_scorer.compute_score({0: item["ground_truth_captions"]}, {0: [item["original_caption"]]})[0]
    )
    scores["original"]["CIDEr"].append(
        cider_scorer.compute_score({0: item["ground_truth_captions"]}, {0: [item["original_caption"]]})[0]
    )

    if idx % 50 == 0 or idx == len(data):
        print(f"✅ Processed {idx}/{len(data)} original captions")

# Print Original Scores
print("\n📊 Original Caption Scores:")
print(f"  BLEU: {avg(scores['original']['BLEU'])}")
print(f"  METEOR: {avg(scores['original']['METEOR'])}")
print(f"  CIDEr: {avg(scores['original']['CIDEr'])}")



🔍 Evaluating Original Captions...
✅ Processed 50/1000 original captions
✅ Processed 100/1000 original captions
✅ Processed 150/1000 original captions
✅ Processed 200/1000 original captions
✅ Processed 250/1000 original captions
✅ Processed 300/1000 original captions
✅ Processed 350/1000 original captions
✅ Processed 400/1000 original captions
✅ Processed 450/1000 original captions
✅ Processed 500/1000 original captions
✅ Processed 550/1000 original captions
✅ Processed 600/1000 original captions
✅ Processed 650/1000 original captions
✅ Processed 700/1000 original captions
✅ Processed 750/1000 original captions
✅ Processed 800/1000 original captions
✅ Processed 850/1000 original captions
✅ Processed 900/1000 original captions
✅ Processed 950/1000 original captions
✅ Processed 1000/1000 original captions

📊 Original Caption Scores:
  BLEU: 0.1811
  METEOR: 0.2617
  CIDEr: 0.0


In [13]:
for atk in attacks:
    print(f"\n⚔️ Evaluating {atk.upper()} Attack...")
    for idx, item in enumerate(data, 1):
        refs = [caption.lower().split() for caption in item["ground_truth_captions"]]
        adv_cap = item[f"{atk}_caption"]
        adv_tok = adv_cap.lower().split()

        scores[atk]["BLEU"].append(sentence_bleu(refs, adv_tok, smoothing_function=smoother))
        scores[atk]["METEOR"].append(
            meteor_scorer.compute_score({0: item["ground_truth_captions"]}, {0: [adv_cap]})[0]
        )
        scores[atk]["CIDEr"].append(
            cider_scorer.compute_score({0: item["ground_truth_captions"]}, {0: [adv_cap]})[0]
        )

        # BERTScore: compare with original
        P, R, F1 = bert_score([adv_cap], [item["original_caption"]], lang="en", verbose=False)
        scores[atk]["BERT"].append(F1.item())

        if idx % 50 == 0 or idx == len(data):
            print(f"✅ Processed {idx}/{len(data)} for {atk}")

    # Print Attack Scores
    print(f"\n📊 Scores under {atk.upper()} Attack:")
    print(f"  BLEU: {avg(scores[atk]['BLEU'])} (drop: {round(avg(scores['original']['BLEU']) - avg(scores[atk]['BLEU']), 4)})")
    print(f"  METEOR: {avg(scores[atk]['METEOR'])} (drop: {round(avg(scores['original']['METEOR']) - avg(scores[atk]['METEOR']), 4)})")
    print(f"  CIDEr: {avg(scores[atk]['CIDEr'])} (drop: {round(avg(scores['original']['CIDEr']) - avg(scores[atk]['CIDEr']), 4)})")
    print(f"  BERTScore F1 vs Original: {avg(scores[atk]['BERT'])}")



⚔️ Evaluating BIM Attack...


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You sho

✅ Processed 50/1000 for bim


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You sho

✅ Processed 100/1000 for bim


...

✅ Processed 900/1000 for bim


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You sho

✅ Processed 950/1000 for bim


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You sho

✅ Processed 1000/1000 for bim

📊 Scores under BIM Attack:
  BLEU: 0.164 (drop: 0.0171)
  METEOR: 0.2291 (drop: 0.0326)
  CIDEr: 0.0 (drop: 0.0)
  BERTScore F1 vs Original: 0.9489
